In [ ]:
import importlib.metadata as _md
_v = _md.version("rocketpy")
if not (_v.startswith("1.10") or _v.startswith("1.11")):
    raise SystemExit(f"Tienes RocketPy {_v}. Este codigo esta probado con 1.10 y 1.11 (la 1.13 calcula mal la masa del "
                     "motor desde el .eng y falla con 'list index out of range'). Instala: pip install rocketpy==1.10.0")
from rocketpy import Environment, GenericMotor, Rocket, Flight

In [ ]:
env = Environment(latitude=41.726083, longitude=0.111083, elevation=192)

In [ ]:
import datetime
env.set_date((2026, 11, 7, 10))  
env.set_atmospheric_model(type="windy", file="ECMWF")

In [ ]:
env.info()

In [ ]:
Viradinha_motor = GenericMotor.load_from_eng_file("datos/motores/AeroTech_H165R.eng")

In [ ]:
Viradinha = Rocket(
    radius=0.0787/2,
    mass=1.271, #sin motor
    inertia=(0.451, 0.451, 0.004),
    power_off_drag="datos/aero/CD_MachOFF.txt",
    power_on_drag="datos/aero/CD_MachOn.txt",
    center_of_mass_without_motor=0.505,
    coordinate_system_orientation="tail_to_nose",
)

In [ ]:
Viradinha.add_motor(Viradinha_motor, position=0.03)

In [ ]:
rail_buttons = Viradinha.set_rail_buttons(
    upper_button_position=0.755,
    lower_button_position=0.085,
    angular_position=45,
)

In [ ]:
nose_cone = Viradinha.add_nose(
    length=0.230, 
    kind="von karman", 
    position=1.23, 
    base_radius=0.0787/2
)

In [ ]:
fin_set = Viradinha.add_trapezoidal_fins(
    n=4,
    root_chord=0.11,
    tip_chord=0.04,
    span=0.072,
    sweep_length=0.07,
    position=0.11,
    airfoil=("datos/aero/NACA0018_Cl_Alpha.txt", "degrees"),
)

In [ ]:
tail = Viradinha.add_tail(
    top_radius=0.0787/2,
    bottom_radius=0.078699/2,
    length=0.05,
    position=0.05
)
    

In [ ]:
# Despliegue por RETARDO DEL MOTOR (8 s desde el apagado), como en el cohete real.
# RocketPy no pasa el tiempo al trigger del paracaidas: se guarda en el calculo de presion
# que RocketPy hace justo antes de llamar al trigger (probado con RocketPy 1.10).
from rocketpy import Flight
_T = {"t": 0.0}
if hasattr(Flight, "_Flight__calculate_and_save_pressure_signals"):
    _orig_pres = Flight._Flight__calculate_and_save_pressure_signals
    def _pres(self, parachute, t, z):
        _T["t"] = t
        return _orig_pres(self, parachute, t, z)
    Flight._Flight__calculate_and_save_pressure_signals = _pres
else:
    raise RuntimeError("Esta version de RocketPy no tiene el gancho; usa RocketPy 1.10 (pip install rocketpy==1.10.0)")

DELAY = 8.0                                   # retardo del motor (s)
T_BURN = Viradinha_motor.burn_out_time        # 1.05 s con el .eng del H165R
def retardo(p, h, y):
    return _T["t"] >= T_BURN + DELAY


In [ ]:
# Paracaidas real: Klima 125 cm, Cd 0.8 -> Cd*S = 0.8 * pi * 0.625^2 = 0.98 m2
import math
main = Viradinha.add_parachute(
    name="Klima 125 cm",
    cd_s=0.8 * math.pi * 0.625**2,
    trigger=retardo,
    sampling_rate=60,
    lag=0.001,
)


In [ ]:
Viradinha.draw()
Viradinha.static_margin.plot()

In [ ]:
test_flight = Flight(rocket=Viradinha, environment=env, rail_length=2.5, inclination=87, heading=0)

In [ ]:
#test_flight.info()

In [ ]:
test_flight.all_info()

In [ ]:
test_flight.export_kml("Viradinh_flight.kml")
test_flight.export_data("Viradinha_flight_data.csv")